# AI-Powered Phishing Email Detection Using NLP and Machine Learning
## Final Project — Baseline v3 (Colab-ready)

This notebook evolves the previous exam project **without changing the scope approved by Prof. Damiano Laviola**:

- binary classification: phishing vs legitimate email;
- text preprocessing;
- TF-IDF feature extraction;
- **one supervised model: Logistic Regression**;
- Accuracy, Precision, Recall, F1-score and ROC-AUC;
- error analysis;
- external validation on an additional corpus from the **Phishing Email Curated Datasets** (Zenodo DOI: `10.5281/zenodo.8339691`).

### Methodological corrections and v3 improvements
1. Split raw text **before** fitting TF-IDF (prevents feature leakage).
2. Use separate **train / validation / test** sets.
3. Tune the optional classification threshold on the **validation set only**, never on the test set.
4. Keep the final test set untouched until final evaluation.
5. Use a source-separated external dataset (TREC_07) to test robustness beyond the original Kaggle aggregate.
6. Add a deterministic **email-address validation layer** for isolated addresses and addresses embedded in message text. This is deliberately **not a second ML model** and is kept separate from the thesis metrics.
7. Add self-tests for obvious provider-domain impersonation such as `gma1l.com` and malformed inputs such as `gma1l,com`.

> The primary academic experiment remains TF-IDF + Logistic Regression. Address validation is an auxiliary input-safety/demo layer and must not be mixed into the reported ML test metrics unless it is evaluated on its own labeled benchmark.

## 1. Install and import dependencies

Run this cell first in Google Colab. The notebook uses only standard NLP/ML libraries and a single classifier.

In [ ]:
!pip -q install -U "kagglehub[pandas-datasets]" "huggingface_hub<2.0"\n

In [ ]:
import os
import re
import json
import hashlib
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports OK")

## 2. Load the original Kaggle dataset

This is the same aggregate used in the previous exam project (`phishing_email.csv`, approximately 82k emails). It remains the primary dataset so the Final Project is a direct continuation of the approved work.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

KAGGLE_DATASET = "naserabdullahalam/phishing-email-dataset"
KAGGLE_FILE = "phishing_email.csv"

print("Loading original Kaggle dataset...")
df_base = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    KAGGLE_DATASET,
    KAGGLE_FILE,
)

required = {"text_combined", "label"}
missing = required - set(df_base.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Available columns: {df_base.columns.tolist()}")

# Keep only the fields required by the approved text-classification scope.
df_base = df_base[["text_combined", "label"]].copy()
df_base = df_base.dropna(subset=["text_combined", "label"])
df_base["text_combined"] = df_base["text_combined"].astype(str)
df_base["label"] = pd.to_numeric(df_base["label"], errors="coerce")
df_base = df_base.dropna(subset=["label"])
df_base["label"] = df_base["label"].astype(int)
df_base = df_base[df_base["label"].isin([0, 1])].copy()

# Deduplicate on a normalized text key and remove contradictory labels for the same text.
def dataset_key(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

df_base["_key"] = df_base["text_combined"].map(dataset_key)
conflicting_keys = (
    df_base.groupby("_key")["label"].nunique().loc[lambda s: s > 1].index
)
conflict_rows = int(df_base["_key"].isin(conflicting_keys).sum())
if conflict_rows:
    df_base = df_base[~df_base["_key"].isin(conflicting_keys)].copy()

before = len(df_base)
df_base = df_base.drop_duplicates(subset=["_key"]).reset_index(drop=True)
duplicates_removed = before - len(df_base)

print(f"Rows after basic cleaning: {before:,}")
print(f"Contradictory-label rows removed: {conflict_rows:,}")
print(f"Normalized duplicates removed: {duplicates_removed:,}")
print(f"Final base rows: {len(df_base):,}")
print(df_base["label"].value_counts().rename(index={0:"Legitimate", 1:"Phishing"}))


## 3. Lightweight text preprocessing

We normalize whitespace and replace URLs / email addresses with tokens instead of deleting them completely. This preserves the fact that an email contains a URL or address while reducing source-specific memorization.

In [ ]:
URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", flags=re.IGNORECASE)
SPACE_RE = re.compile(r"\s+")


def preprocess_text(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" URLTOKEN ", text)
    text = EMAIL_RE.sub(" EMAILTOKEN ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


df_base["processed_text"] = df_base["text_combined"].map(preprocess_text)
df_base = df_base[df_base["processed_text"].str.len() > 0].reset_index(drop=True)
print("Preprocessing complete.")
print(df_base[["processed_text", "label"]].head(2))

## 4. Train / validation / test split — before TF-IDF

- **70% Train**: fit TF-IDF and Logistic Regression.
- **15% Validation**: optionally choose the classification threshold.
- **15% Test**: final untouched evaluation.

The split is stratified to preserve class proportions.

In [ ]:
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    df_base["processed_text"],
    df_base["label"],
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df_base["label"],
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print(f"Train:      {len(X_train_text):,}")
print(f"Validation: {len(X_val_text):,}")
print(f"Test:       {len(X_test_text):,}")

## 5. TF-IDF feature extraction

TF-IDF is fitted **only on the training text**. Validation, test and future emails are transformed using the already-fitted vocabulary.

We use unigrams and bigrams: still the same TF-IDF methodology, but with slightly richer context than the exam notebook's 5,000 unigram features.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=30000,
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train = tfidf_vectorizer.fit_transform(X_train_text)
X_val = tfidf_vectorizer.transform(X_val_text)
X_test = tfidf_vectorizer.transform(X_test_text)

print("TF-IDF fitted on TRAIN only")
print("Train matrix:", X_train.shape)
print("Validation matrix:", X_val.shape)
print("Test matrix:", X_test.shape)

## 6. Train the single supervised baseline: Logistic Regression

In [ ]:
log_reg_model = LogisticRegression(
    solver="liblinear",
    max_iter=1000,
    random_state=RANDOM_STATE,
)
log_reg_model.fit(X_train, y_train)
print("Logistic Regression trained.")

## 7. Validation-set threshold selection

The default threshold is **0.50**. For comparison, we also find the threshold that maximizes F1 on the **validation set only**. The test set is not used during this choice.

In [ ]:
val_proba = log_reg_model.predict_proba(X_val)[:, 1]
precision_v, recall_v, thresholds_v = precision_recall_curve(y_val, val_proba)

# thresholds_v has one fewer element than precision/recall.
f1_v = 2 * precision_v[:-1] * recall_v[:-1] / (precision_v[:-1] + recall_v[:-1] + 1e-12)
best_idx = int(np.nanargmax(f1_v))
optimal_threshold = float(thresholds_v[best_idx])

print(f"Validation-selected threshold (max F1): {optimal_threshold:.4f}")
print(f"Validation F1 at selected threshold: {f1_v[best_idx]:.4f}")

## 8. Final evaluation on the untouched test set

The standard 0.50 threshold remains the primary baseline. The validation-selected threshold is reported as a secondary result.

In [ ]:
def compute_metrics(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
    }, pred


test_proba = log_reg_model.predict_proba(X_test)[:, 1]
metrics_050, y_pred_050 = compute_metrics(y_test, test_proba, 0.50)
metrics_opt, y_pred_opt = compute_metrics(y_test, test_proba, optimal_threshold)

metrics_table = pd.DataFrame([metrics_050, metrics_opt], index=["Test @ 0.50", "Test @ validation threshold"])
display(metrics_table.round(4))

print("\nClassification report @ 0.50")
print(classification_report(y_test, y_pred_050, target_names=["Legitimate", "Phishing"], digits=4))

In [ ]:
# Confusion matrix + ROC + Precision-Recall curves
cm = confusion_matrix(y_test, y_pred_050)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)
ax.set_title("Confusion Matrix — Test @ 0.50")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks([0,1], ["Legitimate", "Phishing"])
ax.set_yticks([0,1], ["Legitimate", "Phishing"])
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center")
plt.show()

fpr, tpr, _ = roc_curve(y_test, test_proba)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, test_proba):.4f}")
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Test Set")
plt.legend()
plt.show()

p, r, _ = precision_recall_curve(y_test, test_proba)
plt.figure(figsize=(6,4))
plt.plot(r, p)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Test Set")
plt.show()

## 9. Error analysis: False Positives and False Negatives

This section provides concrete examples for the Final Report. False Negatives are particularly important because they are phishing emails that the system failed to flag.

In [ ]:
error_df = pd.DataFrame({
    "text": df_base.loc[X_test_text.index, "text_combined"].values,
    "true_label": y_test.values,
    "predicted_label": y_pred_050,
    "phishing_probability": test_proba,
})

false_positives = error_df[(error_df.true_label == 0) & (error_df.predicted_label == 1)].copy()
false_negatives = error_df[(error_df.true_label == 1) & (error_df.predicted_label == 0)].copy()

print(f"False Positives: {len(false_positives):,}")
print(f"False Negatives: {len(false_negatives):,}")

print("\nSample False Positives")
display(false_positives.assign(text=false_positives.text.str.slice(0, 500)).head(5))

print("\nSample False Negatives")
display(false_negatives.assign(text=false_negatives.text.str.slice(0, 500)).head(5))

## 10. Add a genuinely new corpus from the Zenodo collection: TREC_07

The original Kaggle aggregate already reuses several corpora in the Zenodo collection, so concatenating all 11 files would duplicate training examples.

Instead, this notebook adds **TREC_07.csv** as a **source-separated external evaluation set**. It is not used to fit TF-IDF, Logistic Regression, or the threshold. This provides a much more meaningful test of generalization.

Source: **Phishing Email Curated Datasets**, Zenodo, DOI `10.5281/zenodo.8339691`.

The notebook first tries the official Zenodo download. If that fails, it falls back to a byte-identical Hugging Face mirror and verifies the file against Zenodo's published MD5 checksum.

In [ ]:
from huggingface_hub import hf_hub_download
import requests

DATA_DIR = Path("/content/phishing_final_project_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

trec_path = DATA_DIR / "TREC_07.csv"
ZENODO_URL = "https://zenodo.org/records/8339691/files/TREC_07.csv?download=1"
EXPECTED_MD5 = "e5a5ae0de1580965191cb32806dc0dae"
HF_REPO = "kudzaiprichard/aura-phishing-email-corpus"


def md5sum(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def download_zenodo_with_fallback(destination: Path):
    if destination.exists() and md5sum(destination) == EXPECTED_MD5:
        print("TREC_07 already cached and checksum is valid.")
        return destination

    try:
        print("Downloading TREC_07 from Zenodo...")
        with requests.get(ZENODO_URL, stream=True, timeout=(30, 300)) as response:
            response.raise_for_status()
            with open(destination, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        if md5sum(destination) != EXPECTED_MD5:
            raise ValueError("Zenodo download completed but MD5 checksum did not match.")
        print("Zenodo download OK; MD5 verified.")
        return destination
    except Exception as exc:
        print(f"Zenodo download failed ({type(exc).__name__}). Using verified mirror...")
        if destination.exists():
            destination.unlink()
        cached = hf_hub_download(repo_id=HF_REPO, filename="TREC_07.csv", repo_type="dataset")
        shutil.copy2(cached, destination)
        if md5sum(destination) != EXPECTED_MD5:
            raise ValueError("Fallback file checksum does not match the Zenodo record.")
        print("Fallback download OK; Zenodo MD5 verified.")
        return destination


trec_path = download_zenodo_with_fallback(trec_path)
print(trec_path)

In [ ]:
# Load TREC_07 and adapt it to the SAME text-only scope used by the baseline.
df_ext = pd.read_csv(trec_path, low_memory=False, on_bad_lines="skip")
df_ext.columns = [str(c).strip().lower() for c in df_ext.columns]

needed = {"subject", "body", "label"}
missing = needed - set(df_ext.columns)
if missing:
    raise ValueError(f"TREC_07 missing expected columns {missing}. Columns: {df_ext.columns.tolist()}")

# Subject + body only: no sender reputation, headers or URL metadata are added to the model.
df_ext["subject"] = df_ext["subject"].fillna("").astype(str)
df_ext["body"] = df_ext["body"].fillna("").astype(str)
df_ext["text_combined"] = (df_ext["subject"] + " " + df_ext["body"]).str.strip()
df_ext["label"] = pd.to_numeric(df_ext["label"], errors="coerce")
df_ext = df_ext.dropna(subset=["label"])
df_ext["label"] = df_ext["label"].astype(int)
df_ext = df_ext[df_ext["label"].isin([0, 1])]
df_ext = df_ext[df_ext["text_combined"].str.len() > 0].copy()

# Normalize and deduplicate both internally and against the original dataset.
def dedup_key(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

base_keys = set(df_base["_key"])
df_ext["_key"] = df_ext["text_combined"].map(dedup_key)

before = len(df_ext)
df_ext = df_ext.drop_duplicates(subset=["_key", "label"])
internal_removed = before - len(df_ext)

overlap_mask = df_ext["_key"].isin(base_keys)
overlap_removed = int(overlap_mask.sum())
df_ext = df_ext.loc[~overlap_mask].copy()
df_ext["processed_text"] = df_ext["text_combined"].map(preprocess_text)

print(f"TREC_07 rows loaded: {before:,}")
print(f"Internal duplicates removed: {internal_removed:,}")
print(f"Overlaps with original Kaggle aggregate removed: {overlap_removed:,}")
print(f"External evaluation rows retained: {len(df_ext):,}")
print(df_ext["label"].value_counts().rename(index={0:"Legitimate",1:"Phishing"}))

## 11. External robustness evaluation

This is intentionally harder than a random split: TREC_07 is kept outside training. A lower score here is not a failure of the project — it is valuable evidence about real generalization and dataset bias.

In [ ]:
X_ext = tfidf_vectorizer.transform(df_ext["processed_text"])
y_ext = df_ext["label"].to_numpy()
ext_proba = log_reg_model.predict_proba(X_ext)[:, 1]

ext_050, ext_pred_050 = compute_metrics(y_ext, ext_proba, 0.50)
ext_opt, ext_pred_opt = compute_metrics(y_ext, ext_proba, optimal_threshold)

external_metrics = pd.DataFrame([ext_050, ext_opt], index=["TREC_07 @ 0.50", "TREC_07 @ validation threshold"])
display(external_metrics.round(4))

print("\nExternal classification report @ 0.50")
print(classification_report(y_ext, ext_pred_050, target_names=["Legitimate", "Phishing"], digits=4))

## 12. Robust prediction: full email text **or** an isolated email address

The ML classifier was trained on **email text**, so an isolated address must not be forced through the text model.

V3 therefore uses two clearly separated paths:

1. **Full subject/body text** → TF-IDF + Logistic Regression (the approved ML baseline).
2. **Address-only input** → deterministic syntax + provider-domain anomaly checks.

The address checker looks for malformed domains, digit/letter substitutions and typo-squatting near common providers (for example `gma1l.com` → `gmail.com`). A syntactically valid address is **not automatically trustworthy**; the checker only reports address-level anomalies.

For a full message, the notebook also scans embedded addresses and reports suspicious address indicators alongside the ML prediction.

In [ ]:
# -----------------------------------------------------------------------------
# V3 address validation layer
# -----------------------------------------------------------------------------
# IMPORTANT FOR THE FINAL REPORT:
# - This is deterministic input validation, NOT a second supervised model.
# - Primary thesis metrics remain those of TF-IDF + Logistic Regression.
# - A valid address format does not prove that the sender/mailbox is trustworthy.

STRICT_EMAIL_RE = re.compile(
    r"^[A-Z0-9.!#$%&'*+/=?^_`{|}~-]+@(?:[A-Z0-9](?:[A-Z0-9-]{0,61}[A-Z0-9])?\.)+[A-Z]{2,63}$",
    flags=re.IGNORECASE,
)

# Common providers and selected high-value domains: used ONLY to detect close impersonation/typosquatting.
# Other legitimate custom domains are not marked suspicious merely because they are absent here.
COMMON_PROVIDER_DOMAINS = {
    "gmail.com", "googlemail.com",
    "outlook.com", "hotmail.com", "live.com",
    "yahoo.com", "yahoo.it",
    "icloud.com", "me.com",
    "proton.me", "protonmail.com",
    "aol.com", "zoho.com", "gmx.com", "gmx.net",
    "libero.it", "virgilio.it", "alice.it", "tin.it",
    "tiscali.it", "tim.it", "fastwebnet.it", "aruba.it",
    # High-value domains frequently impersonated in phishing campaigns.
    "paypal.com", "amazon.com", "apple.com", "microsoft.com", "google.com",
    "poste.it", "intesasanpaolo.com", "unicredit.it",
}

# Digits commonly used as visual substitutions in phishing/typosquatting domains.
LEET_OPTIONS = {
    "0": ("o",),
    "1": ("i", "l"),
    "3": ("e",),
    "4": ("a",),
    "5": ("s",),
    "7": ("t",),
    "8": ("b",),
    "9": ("g",),
}


def _damerau_levenshtein(a: str, b: str) -> int:
    """Small-string Damerau-Levenshtein distance (insert/delete/substitute/transpose)."""
    a, b = a.lower(), b.lower()
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        d[i][0] = i
    for j in range(len(b) + 1):
        d[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            d[i][j] = min(
                d[i - 1][j] + 1,
                d[i][j - 1] + 1,
                d[i - 1][j - 1] + cost,
            )
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                d[i][j] = min(d[i][j], d[i - 2][j - 2] + 1)
    return d[-1][-1]


def _leet_domain_variants(domain: str, max_variants: int = 64):
    """Generate a small set of letter-normalized variants for digit-containing domains."""
    variants = {domain.lower()}
    for ch, replacements in LEET_OPTIONS.items():
        if ch not in domain:
            continue
        updated = set(variants)
        for item in variants:
            for repl in replacements:
                updated.add(item.replace(ch, repl))
                if len(updated) >= max_variants:
                    break
            if len(updated) >= max_variants:
                break
        variants = set(list(updated)[:max_variants])
    return variants


def _nearest_common_provider(domain: str):
    """Return (provider, distance, via_leet) when a common-provider lookalike is found."""
    raw = domain.lower()

    # Exact provider = no typo signal.
    if raw in COMMON_PROVIDER_DOMAINS:
        return raw, 0, False

    # First catch classic digit/letter substitutions (gma1l.com -> gmail.com).
    for variant in _leet_domain_variants(raw):
        if variant != raw and variant in COMMON_PROVIDER_DOMAINS:
            return variant, 0, True

    # Then catch one-edit / one-transposition typos (gmial.com, gmai.com, gmail.co etc.).
    best_provider = None
    best_distance = 999
    for provider in COMMON_PROVIDER_DOMAINS:
        dist = _damerau_levenshtein(raw, provider)
        if dist < best_distance:
            best_provider, best_distance = provider, dist
    if best_distance <= 1:
        return best_provider, best_distance, False

    return None, None, False


def _looks_like_address_only(raw: str) -> bool:
    """Route compact @-containing inputs to address validation even when malformed."""
    raw = str(raw).strip()
    return (
        "@" in raw
        and "\n" not in raw
        and "\r" not in raw
        and not re.search(r"\s", raw)
        and len(raw) <= 320
    )


def analyze_email_address(address: str) -> dict:
    original = str(address).strip().strip("<>")
    reasons = []
    warnings_list = []

    if not original:
        return {
            "address": original,
            "syntax_valid": False,
            "risk": "INVALID_OR_SUSPICIOUS",
            "reasons": ["Empty address."],
            "closest_provider": None,
        }

    if original.count("@") != 1:
        return {
            "address": original,
            "syntax_valid": False,
            "risk": "INVALID_OR_SUSPICIOUS",
            "reasons": ["An email address must contain exactly one @ symbol."],
            "closest_provider": None,
        }

    local, domain = original.rsplit("@", 1)
    domain_lower = domain.lower().rstrip(".")

    # Diagnostic-only normalization: never silently fix the address.
    diagnostic_domain = domain_lower.replace(",", ".").replace(";", ".")

    if "," in domain or ";" in domain:
        reasons.append("The domain contains a comma/semicolon instead of a dot.")

    syntax_valid = bool(STRICT_EMAIL_RE.fullmatch(original))
    if not syntax_valid:
        reasons.append("The address does not match a valid standard email-domain format.")

    if not local:
        reasons.append("Missing local part before @.")
    if not domain:
        reasons.append("Missing domain after @.")

    # Non-ASCII / IDN can be legitimate, so treat it as a review warning rather than proof of phishing.
    if any(ord(ch) > 127 for ch in original):
        warnings_list.append("Non-ASCII characters detected; review for possible Unicode homoglyphs.")
    if domain_lower.startswith("xn--") or ".xn--" in domain_lower:
        warnings_list.append("Punycode/IDN domain detected; manual verification is recommended.")

    # Provider lookalike checks are performed on the raw domain and, if malformed punctuation
    # exists, on a diagnostic dot-normalized version too.
    candidate_domains = [domain_lower]
    if diagnostic_domain != domain_lower:
        candidate_domains.append(diagnostic_domain)

    closest_provider = None
    provider_signal = None
    for candidate in candidate_domains:
        provider, distance, via_leet = _nearest_common_provider(candidate)
        if provider and candidate != provider:
            closest_provider = provider
            if via_leet:
                provider_signal = (
                    f"Digit/letter substitution makes domain '{candidate}' look like '{provider}'."
                )
            else:
                provider_signal = (
                    f"Domain '{candidate}' is only {distance} edit away from common provider '{provider}'."
                )
            break

    if provider_signal:
        reasons.append(provider_signal)

    # Digits in the domain are not automatically malicious (many real domains contain them),
    # but they become high-risk when they create a lookalike of a known provider.
    if any(ch.isdigit() for ch in domain_lower) and closest_provider:
        reasons.append("A digit is used inside a provider lookalike domain.")

    # Structural domain checks (use diagnostic domain to provide a useful explanation on comma typos).
    labels = diagnostic_domain.split(".") if diagnostic_domain else []
    if any(label == "" for label in labels):
        reasons.append("The domain contains an empty label / repeated separator.")
    for label in labels:
        if label.startswith("-") or label.endswith("-"):
            reasons.append("A domain label starts or ends with a hyphen.")
            break

    if reasons:
        risk = "INVALID_OR_SUSPICIOUS"
    elif warnings_list:
        risk = "REVIEW"
    else:
        risk = "NO_OBVIOUS_ADDRESS_ANOMALY"

    return {
        "address": original,
        "syntax_valid": syntax_valid,
        "risk": risk,
        "reasons": reasons,
        "warnings": warnings_list,
        "closest_provider": closest_provider,
    }


ADDRESS_CANDIDATE_RE = re.compile(
    r"[A-Z0-9.!#$%&'*+/=?^_`{|}~-]+@[^\s<>\"']+",
    flags=re.IGNORECASE,
)


def extract_address_candidates(text: str):
    candidates = []
    for match in ADDRESS_CANDIDATE_RE.findall(str(text)):
        # Remove punctuation that commonly follows an address in prose, but retain punctuation
        # inside the domain (e.g. gma1l,com) so the analyzer can flag it.
        candidate = match.strip("()[]{}<>\"'")
        candidate = re.sub(r"[.!?:]+$", "", candidate)
        if candidate.endswith(",") or candidate.endswith(";"):
            candidate = candidate[:-1]
        if "@" in candidate:
            candidates.append(candidate)
    # preserve order, remove duplicates
    return list(dict.fromkeys(candidates))


def inspect_addresses_in_text(text: str):
    findings = []
    for address in extract_address_candidates(text):
        result = analyze_email_address(address)
        if result["risk"] != "NO_OBVIOUS_ADDRESS_ANOMALY":
            findings.append(result)
    return findings


def predict_email_phishing(email_text, threshold=0.50):
    raw = str(email_text).strip()

    if not raw:
        return {"mode": "none", "status": "insufficient_input", "message": "No input entered."}

    # Address-only inputs: do NOT force them through the text classifier.
    if _looks_like_address_only(raw):
        address_result = analyze_email_address(raw)
        return {
            "mode": "address_only",
            "status": "ok",
            "address_analysis": address_result,
        }

    # For actual message text, require a minimum amount of linguistic content.
    if len(re.findall(r"\b\w+\b", raw)) < 5:
        return {
            "mode": "text",
            "status": "insufficient_input",
            "message": "Too little message text for a reliable TF-IDF classification. Paste a longer subject/body fragment.",
        }

    processed = preprocess_text(raw)
    features = tfidf_vectorizer.transform([processed])
    probability = float(log_reg_model.predict_proba(features)[0, 1])
    ml_label = "Phishing" if probability >= threshold else "Legitimate"

    address_findings = inspect_addresses_in_text(raw)
    high_risk_address = any(
        item["risk"] == "INVALID_OR_SUSPICIOUS" for item in address_findings
    )

    # Keep the ML prediction explicit and separate from deterministic indicators.
    overall = "SUSPICIOUS" if (ml_label == "Phishing" or high_risk_address) else "NO_OBVIOUS_PHISHING_SIGNAL"

    return {
        "mode": "full_email_text",
        "status": "ok",
        "ml_prediction": ml_label,
        "phishing_probability": probability,
        "threshold": float(threshold),
        "suspicious_address_findings": address_findings,
        "overall_screening": overall,
    }


def print_prediction_result(result: dict):
    if result.get("status") != "ok":
        print(result.get("message", "Unable to analyze input."))
        return

    if result["mode"] == "address_only":
        a = result["address_analysis"]
        print("Mode: isolated email-address validation")
        print(f"Address: {a['address']}")
        print(f"Syntax valid: {a['syntax_valid']}")
        print(f"Address verdict: {a['risk']}")
        if a.get("closest_provider"):
            print(f"Possible intended provider: {a['closest_provider']}")
        for reason in a.get("reasons", []):
            print(" -", reason)
        for warning in a.get("warnings", []):
            print(" - Warning:", warning)
        if a["risk"] == "NO_OBVIOUS_ADDRESS_ANOMALY":
            print("Note: valid-looking syntax does not prove the mailbox or sender is trustworthy.")
        return

    print("Mode: full email-text phishing classification")
    print(f"ML prediction: {result['ml_prediction']}")
    print(f"Phishing probability: {result['phishing_probability']:.2%}")
    print(f"Overall screening: {result['overall_screening']}")
    findings = result.get("suspicious_address_findings", [])
    if findings:
        print("Suspicious address indicators found:")
        for finding in findings:
            print(f" * {finding['address']} -> {finding['risk']}")
            for reason in finding.get("reasons", []):
                print("    -", reason)


### 12.1 Self-test of the address validator

These checks prevent an obvious regression in the demo layer. They do **not** replace a labeled evaluation dataset; they simply verify the intended behavior on a few deterministic cases.

In [ ]:
ADDRESS_SELF_TESTS = [
    ("gcasaldi@gmail.com", "NO_OBVIOUS_ADDRESS_ANOMALY"),
    ("test@outlook.com", "NO_OBVIOUS_ADDRESS_ANOMALY"),
    ("hot@gma1l.com", "INVALID_OR_SUSPICIOUS"),
    ("hot@gma1l,com", "INVALID_OR_SUSPICIOUS"),
    ("support@paypa1.com", "INVALID_OR_SUSPICIOUS"),
    ("security@gmial.com", "INVALID_OR_SUSPICIOUS"),
]

for address, expected in ADDRESS_SELF_TESTS:
    actual = analyze_email_address(address)["risk"]
    print(f"{address:28s} -> {actual}")
    assert actual == expected, f"Self-test failed for {address}: expected {expected}, got {actual}"

print("\nAll V3 address-validation self-tests passed.")

### 12.2 Interactive demo

Paste either:
- a **complete email address**, or
- a **subject/body fragment** with at least a few words.

The output explicitly tells you which analysis path was used.

In [ ]:
print("--- V3 phishing / address analysis ---")
user_input = input("Paste an email address OR the email subject/body text: ")
result = predict_email_phishing(user_input, threshold=0.50)
print()
print_prediction_result(result)

## 13. Save reproducible outputs

This creates model artifacts and metric files that can be referenced in the Final Project. The files can be downloaded from the Colab file panel after a successful run.

In [ ]:
import joblib

OUT_DIR = Path("/content/final_project_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(tfidf_vectorizer, OUT_DIR / "tfidf_vectorizer.joblib")
joblib.dump(log_reg_model, OUT_DIR / "logistic_regression.joblib")

all_metrics = {
    "kaggle_test_threshold_0_50": metrics_050,
    "kaggle_test_validation_selected_threshold": metrics_opt,
    "external_trec07_threshold_0_50": ext_050,
    "external_trec07_validation_selected_threshold": ext_opt,
    "validation_selected_threshold": optimal_threshold,
    "random_state": RANDOM_STATE,
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

false_positives.to_csv(OUT_DIR / "false_positives.csv", index=False)
false_negatives.to_csv(OUT_DIR / "false_negatives.csv", index=False)

print("Saved outputs:")
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name)

# Save V3 address-validator configuration for reproducibility/documentation.
address_validation_config = {
    "common_provider_domains": sorted(COMMON_PROVIDER_DOMAINS),
    "leet_options": {k: list(v) for k, v in LEET_OPTIONS.items()},
    "note": "Deterministic auxiliary validation layer; not included in primary ML metrics.",
}
with open(OUT_DIR / "address_validation_config.json", "w") as f:
    json.dump(address_validation_config, f, indent=2)

print(" - address_validation_config.json")

## 14. Notes for the Final Report

The core experiment remains intentionally simple and aligned with the approved proposal:

**Email text → preprocessing → TF-IDF → Logistic Regression → Accuracy / Precision / Recall / F1 / ROC-AUC.**

The academically important improvements are methodological rather than model proliferation:

- no TF-IDF leakage;
- no test-set threshold tuning;
- explicit duplicate control;
- source-separated external evaluation;
- False Positive / False Negative analysis;
- external TREC_07 evaluation from Zenodo;
- explicit handling of out-of-scope inputs;
- deterministic address validation for malformed/typosquatted provider domains.

### Important interpretation rule
The address validator is an **auxiliary deterministic screening layer**, not part of the supervised ML baseline. Do not combine its handcrafted self-tests with the Logistic Regression Accuracy/F1 values in the Final Report. If the address layer becomes a research contribution in its own right, it should later receive a separate labeled benchmark.

A result such as `NO_OBVIOUS_ADDRESS_ANOMALY` means only that the address has valid-looking syntax and no obvious provider lookalike detected by these rules; it does **not** prove sender identity or trustworthiness.

### Dataset citation
A. I. Champa, M. F. Rabbi, and M. F. Zibran, *Curated Datasets and Feature Analysis for Phishing Email Detection with Machine Learning*, 3rd IEEE International Conference on Computing and Machine Intelligence (ICMI), 2024.

Zenodo record: `10.5281/zenodo.8339691`.